# DisloCluster — run a coupled simulation

Set up and run an operator-split 0-D ↔ 3-D cluster-dynamics simulation of
irradiated zirconium.

Everything you normally change lives in **§2**: material, geometry, mesh,
boundary conditions, coupling and solver settings, and what to output.
Nothing below §2 should need editing for an ordinary run.

| Section | What it does |
|---|---|
| 1 | Environment and builds |
| 2 | **The control dicts — edit these** |
| 3 | Validate the configuration |
| 4 | Mesh, staged case, CD node set, seed |
| 5 | The march (long; resumable) |
| 6 | Results |
| 7 | Figures, movies, provenance |

**The long cell is §5.** It checkpoints after every substep, so if the kernel
dies or you interrupt it, just run the cell again — it picks up where it
stopped rather than starting over.

## 1. Environment

In [3]:
# Every location is resolved from the .dislocluster_root marker at the top of
# the repository, so this works from any checkout on any machine -- nothing
# below depends on where the repository lives or on who built it last.
from dislocluster_code import paths, driver
from dislocluster_code.config import SimulationConfig

print(paths.describe())

OK   repo root           /Users/ghoni/Documents/GitHub/DisloCluster
OK   venv                /Users/ghoni/Documents/GitHub/DisloCluster/.DisloClusterVenv
OK   0-D code            /Users/ghoni/Documents/GitHub/DisloCluster/ZrMicro
OK   0-D solver          /Users/ghoni/Documents/GitHub/DisloCluster/ZrMicro/build/solver
OK   input               /Users/ghoni/Documents/GitHub/DisloCluster/Simulations/input
OK   output (writes)     /Users/ghoni/Documents/GitHub/DisloCluster/Simulations/output
OK   output (also read)  /Users/ghoni/Documents/GitHub/DisloCluster/ZrMicro/output
OK   MoDELib root        /Users/ghoni/Documents/GitHub/DisloCluster/MoDELib3
OK   MoDELib build       /Users/ghoni/Documents/GitHub/DisloCluster/MoDELib3/build
OK   MoDELib DDomp       /Users/ghoni/Documents/GitHub/DisloCluster/MoDELib3/build/tools/DDomp/DDomp
OK   Zr material         /Users/ghoni/Documents/GitHub/DisloCluster/MoDELib3/Library/Materials/Zr3d_ghoniem.txt
OK   coupled sim case    /Users/ghoni/Documents/GitH

In [4]:
# One call makes this machine ready, or explains exactly what is missing.
#
# It resolves every path from the repository marker, builds either C++ code if
# its binary is absent, discards a CMake cache left pointing at a previous
# location, and then RUNS both binaries to prove they load -- existing on disk
# is not the same as being runnable. Neither build tree is in git, so a fresh
# clone builds its own; the MoDELib build runs inside WSL and takes 10-30 min
# the first time, and is a no-op afterwards.
#
# It raises with actionable text if something cannot be fixed automatically
# (SUNDIALS not installed, WSL unavailable, no compiler).
from dislocluster_code import build

build.preflight()

DisloCluster preflight
OK   repo root           /Users/ghoni/Documents/GitHub/DisloCluster
OK   venv                /Users/ghoni/Documents/GitHub/DisloCluster/.DisloClusterVenv
OK   0-D code            /Users/ghoni/Documents/GitHub/DisloCluster/ZrMicro
OK   0-D solver          /Users/ghoni/Documents/GitHub/DisloCluster/ZrMicro/build/solver
OK   input               /Users/ghoni/Documents/GitHub/DisloCluster/Simulations/input
OK   output (writes)     /Users/ghoni/Documents/GitHub/DisloCluster/Simulations/output
OK   output (also read)  /Users/ghoni/Documents/GitHub/DisloCluster/ZrMicro/output
OK   MoDELib root        /Users/ghoni/Documents/GitHub/DisloCluster/MoDELib3
OK   MoDELib build       /Users/ghoni/Documents/GitHub/DisloCluster/MoDELib3/build
OK   MoDELib DDomp       /Users/ghoni/Documents/GitHub/DisloCluster/MoDELib3/build/tools/DDomp/DDomp
OK   Zr material         /Users/ghoni/Documents/GitHub/DisloCluster/MoDELib3/Library/Materials/Zr3d_ghoniem.txt
OK   coupled sim case    /Use

{'solver': '/Users/ghoni/Documents/GitHub/DisloCluster/ZrMicro/build/solver',
 'ddomp': '/Users/ghoni/Documents/GitHub/DisloCluster/MoDELib3/build/tools/DDomp/DDomp',
 'drift': []}

## 2. The control dicts

These are the whole user interface. Anything absent falls back to the default
in `dislocluster_code.config`, and a key that is not a real setting raises rather
than being silently ignored.

In [5]:
MATERIAL = {
    "file":            "Zr3d_ghoniem.txt",   # MoDELib3/Library/Materials/
    "workbook":        None,                 # None -> the calibrated default
    "temperature_K":   573.0,
    "dose_rate_dpa_s": 1.0e-7,
    # Extra 0-D parameters, applied ON TOP OF the fitted set in
    # zerod/calibration.py. Leave empty unless you mean to move the model
    # away from its calibration -- the workbook alone is NOT the calibrated
    # model (28 parameters have drifted, 11 are missing).
    "overrides":       {},
}

In [6]:
GEOMETRY = {
    "type":      "cubic",        # "cubic" | "hexagonal"
    "size_nm":   500.0,          # cube side, or hexagon corner-to-corner
    "height_nm": None,           # hexagonal only; None -> size_nm
    "aspect":    (1.0, 1.0, 1.0),
}

MESH = {
    "target_elements":        15000,   # sets the interior element size
    "lc_nm":                  120.0,   # explicit interior size; wins if set
    "element_order":          1,       # 1 = linear tet, 2 = quadratic
    # A refined shell resolves the boundary layer, where the mobile species
    # are pinned to thermal equilibrium and fall over a few tens of nm.
    "boundary_layer":         True,
    "boundary_thickness_nm":  150.0,
    "boundary_lc_nm":         None,    # or set the layer size outright
    "boundary_layers_across": 5,       # layer size = thickness / this
    "boundary_size_ratio":    0.35,
    "boundary_expansion":     None,
    "optimize":               True,
    "regenerate":             False,   # force a rebuild even on a cache hit
}

In [7]:
BOUNDARY = {
    # Which faces are periodic. [] means none, i.e. Dirichlet on the whole
    # surface -- the mobile species are held at thermal equilibrium there and
    # the surface acts as a sink.
    "periodic_face_ids":   [],

    # Mechanical load, Voigt order 11, 22, 33, 12, 23, 13.
    # Given in MPa here; MoDELib normalizes stress by the shear modulus and
    # the conversion is done for you when the case is staged.
    "applied_stress_MPa":  (0.0,) * 6,
    "applied_stress_rate_MPa_s": (0.0,) * 6,
    "applied_strain":      (0.0,) * 6,
    "applied_strain_rate": (0.0,) * 6,
    # 0 = pure stress control, 1e20 = pure strain control, per component
    "stiffness_ratio":     (0.0,) * 6,
}

In [8]:
COUPLING = {
    "route":     "Adaptive",     # "Adaptive" (CVODE BDF) | "IMEX" (MoDELib nodal)

    # Where the march starts. 0 or less seeds PRISTINE material, which is the
    # cleanest choice: mobile transients settle by 1e-4 dpa while loop
    # nucleation runs over 1e-3 to 1e-2 dpa, so seeding earlier moves further
    # into the regime where the operator split is valid.
    "dose_seed": 0.0,

    # The snapshot doses. A logarithmic grid through the nucleation transient
    # is fine -- the grid does not have to lie on the 1 dpa lattice.
    "doses":     [1.0, 6.0, 11.0, 16.0, 21.0],

    "substeps_per_interval": 20,
    # How many immobile substeps between two fast solves. The splitting error
    # is first order in THIS spacing, not in the snapshot spacing. Each fast
    # solve is a separate DDomp invocation of a few minutes, so this is the
    # main accuracy-versus-wall-clock dial.
    "fem_every":             5,

    "dedup_rtol":       1e-8,
    "variant_weights":  (1/3, 1/3, 1/3),   # <a> variants; equal at zero stress
    "max_failed_nodes": 0,      # abort if any node's integration fails
    "on_unconverged":   "warn", # "raise" to refuse a capped mobile solve
}

In [9]:
SOLVER = {
    "rtol": 1e-6, "atol": 1e-20,
    "backend": "cvode", "lmm": "bdf", "linsol": "dense",
    "analytic_jac": True,
}

OUTPUT = {
    "tag":            None,      # None -> derived from the case name
    "figures":        True,
    "movies":         False,     # slow
    "discrete_loops": False,     # slower: samples the continuum field into loops
    "checkpoint":     True,      # required for resume
    "resume":         "auto",    # "auto" | "never" | "require"
}

## 3. Validate

Everything is checked here, before anything expensive happens: a dose grid
that does not increase, a boundary layer thicker than the domain, an element
order MoDELib cannot read, a mistyped key.

In [10]:
cfg = SimulationConfig.from_dicts(
    MATERIAL=MATERIAL, GEOMETRY=GEOMETRY, MESH=MESH, BOUNDARY=BOUNDARY,
    COUPLING=COUPLING, SOLVER=SOLVER, OUTPUT=OUTPUT)

print(cfg.summary())

case        cubic_500nm_o1_647246e130631c45
geometry    cubic, 500 nm
mesh        order 1, ~15000 elements, 150 nm boundary layer, 5 across
material    Zr3d_ghoniem.txt at 573 K, 1e-07 dpa/s
boundary    Dirichlet on every face; zero applied stress
route       Option B — Adaptive: SUNDIALS CVODE, adaptive-order adaptive-step BDF
dose        seed 0 -> 1, 6, 11, 16, 21 dpa
march       20 substeps/interval, fast solve every 5 (100 substeps total)
sim dir     /Users/ghoni/Documents/GitHub/DisloCluster/MoDELib3/tutorials/cubic_500nm_o1_647246e130631c45


## 4. Prepare

Generates the mesh, stages the MoDELib case, and runs the one bootstrap call
that discovers the CD node set — the cluster-dynamics trial functions live on
second-order elements, so the node set is not the mesh vertex list and only
MoDELib knows it.

This is skipped when a staged case already matches the configuration, so
changing only the dose grid does not re-mesh.

In [12]:
run = driver.prepare(cfg)

case        cubic_500nm_o1_647246e130631c45
geometry    cubic, 500 nm
mesh        order 1, ~15000 elements, 150 nm boundary layer, 5 across
material    Zr3d_ghoniem.txt at 573 K, 1e-07 dpa/s
boundary    Dirichlet on every face; zero applied stress
route       Option B — Adaptive: SUNDIALS CVODE, adaptive-order adaptive-step BDF
dose        seed 0 -> 1, 6, 11, 16, 21 dpa
march       20 substeps/interval, fast solve every 5 (100 substeps total)
sim dir     /Users/ghoni/Documents/GitHub/DisloCluster/MoDELib3/tutorials/cubic_500nm_o1_647246e130631c45

domain up to date: cubic_500nm_o1_647246e130631c45 (31743 CD nodes) — staging skipped
InputData: Attempting to load /Users/ghoni/Documents/GitHub/DisloCluster/Simulations/input/Zr_input_parameters.xlsx
✓ File exists: /Users/ghoni/Documents/GitHub/DisloCluster/Simulations/input/Zr_input_parameters.xlsx
Reading Excel file: /Users/ghoni/Documents/GitHub/DisloCluster/Simulations/input/Zr_input_parameters.xlsx
✓ Successfully loaded input data from

## 5. The march

**The long cell.** It writes a checkpoint after every substep, so re-running
this cell after an interrupt or a kernel restart continues from the last
completed substep instead of starting over.

Set `OUTPUT['resume'] = 'never'` above to force a fresh start.

In [ ]:
result = driver.march(run)

## 6. Results

In [ ]:
driver.print_table(result)
print(f"\nwall clock {result.wall_s/3600:.2f} h")
print(f"output     {result.out_dir}")

In [ ]:
# Where the time went, and whether the fast solve behaved.
d = result.diagnostics
if "fast_solver" in d:
    f = d["fast_solver"]
    print(f"fast solves : {f['calls']}, {f['wall_s']:.0f} s "
          f"({f['mean_s']:.0f} s each)")
if "checkpoint" in d:
    c = d["checkpoint"]
    print(f"checkpoints : {c['writes']} writes, "
          f"{c['mean_write_ms']:.0f} ms each"
          + ("  (this run resumed)" if c["resumed"] else ""))
for t in result.timing:
    print(f"  {t['dose_from']:8.4g} -> {t['dose_to']:8.4g} dpa  "
          f"{t['wall_s']:8.0f} s  "
          f"({t['fast_s']:.0f} s fast / {t['slow_s']:.0f} s slow, "
          f"{t['fast_solves']} FEM)")

## 7. Figures, movies, provenance

Writes `3d/`, `gb/`, `volume_average/` and `report.md` into the run directory.

Two things to read carefully in the output:

- **Near-boundary loop densities are not quantitative.** Cascade nucleation is
  spatially uniform, but the only loop-removal channel is coalescence, driven
  by the absorbed mobile flux — which vanishes where Dirichlet pins the mobile
  concentrations. The boundary is a sink for mobile defects but *not* for
  loops, so ⟨a⟩ density climbs steeply toward it.
- **Never quote a whole-domain mean of a loop quantity on this geometry.** The
  surface nodes carry almost all of the total ⟨a⟩ density. Quote the interior
  mean instead; `report.md` does.

In [ ]:
driver.report(run, result)